# 1. Check the Runtime and GPU

Inspect the Python environment, PyTorch installation, and available GPU before installing dependencies. A GPU such as a Tesla T4 is useful for this small fine-tuning experiment.

In [ ]:
import sys
import platform

print("Python version:", sys.version)
print("Platform:", platform.platform())

try:
    import torch
    print("PyTorch version:", torch.__version__)
    print("GPU available:", torch.cuda.is_available())

    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
    else:
        print("No GPU detected. In Colab, select Runtime → Change runtime type → T4 GPU if available.")
except ImportError:
    print("PyTorch is not installed yet.")

Python version: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
Platform: Linux-6.6.122+-x86_64-with-glibc2.39
PyTorch version: 2.11.0+cu130
GPU available: True
GPU: Tesla T4


# 2. Install Project Dependencies

Install the Python packages used for data handling, model loading, LoRA fine-tuning, training, and the Gradio demo. Run this cell in Google Colab.

In [ ]:
%pip install -q "pandas==2.2.3" "transformers" "peft" "datasets" "accelerate" "gradio" "safetensors"

# 3. Verify Installed Package Versions

Import the required libraries and print their versions. This helps identify dependency conflicts before training.

In [ ]:
import torch
import pandas as pd
import transformers
import peft
import datasets
import accelerate
import gradio

print("Pandas:", pd.__version__)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("Datasets:", datasets.__version__)
print("Accelerate:", accelerate.__version__)
print("Gradio:", gradio.__version__)
print("GPU available:", torch.cuda.is_available())

Pandas: 2.2.3
Transformers: 5.20.2
PEFT: 0.21.2
Datasets: 4.8.5
Accelerate: 1.15.0
Gradio: 6.30.0
GPU available: True


# 4. Create the Bug-Explanation Dataset

Build a small synthetic dataset of Python bug examples, explanations, and corrected code. The dataset is intended for a proof-of-concept only; it is too small to establish reliable real-world performance.

In [ ]:
import json
import random
import pandas as pd
from sklearn.model_selection import train_test_split

random.seed(42)

examples = [
    {
        "bug": "def add(a, b):\n    return a - b",
        "explanation": "The function subtracts instead of adding. Replace the minus operator with plus.",
        "fixed_code": "def add(a, b):\n    return a + b"
    },
    {
        "bug": "items = [1, 2, 3]\nprint(items[3])",
        "explanation": "Python lists use zero-based indexing. A three-element list has valid indices 0, 1, and 2. Use items[2] to access the last element.",
        "fixed_code": "items = [1, 2, 3]\nprint(items[2])"
    },
    {
        "bug": "if x = 5:\n    print('five')",
        "explanation": "A single equals sign assigns a value and cannot be used for comparison in an if condition. Use == to compare values.",
        "fixed_code": "if x == 5:\n    print('five')"
    },
    {
        "bug": "for i in range(5)\n    print(i)",
        "explanation": "The for statement is missing a colon at the end of the loop header.",
        "fixed_code": "for i in range(5):\n    print(i)"
    },
    {
        "bug": "def greet(name):\nprint('Hello', name)",
        "explanation": "The function body must be indented beneath the function definition.",
        "fixed_code": "def greet(name):\n    print('Hello', name)"
    },
    {
        "bug": "name = 'Alex'\nprint(Name)",
        "explanation": "Python variable names are case-sensitive. Name and name are different identifiers.",
        "fixed_code": "name = 'Alex'\nprint(name)"
    },
    {
        "bug": "numbers = [1, 2, 3]\nnumbers.append[4]",
        "explanation": "append is a method and must be called with parentheses, not square brackets.",
        "fixed_code": "numbers = [1, 2, 3]\nnumbers.append(4)"
    },
    {
        "bug": "def divide(a, b):\n    return a / b\n\nprint(divide(10, 0))",
        "explanation": "Dividing by zero raises ZeroDivisionError. Check whether the denominator is zero before dividing.",
        "fixed_code": "def divide(a, b):\n    if b == 0:\n        raise ValueError('b must not be zero')\n    return a / b"
    },
    {
        "bug": "text = 'hello\nprint(text)",
        "explanation": "The string literal is missing its closing quotation mark.",
        "fixed_code": "text = 'hello'\nprint(text)"
    },
    {
        "bug": "total = 0\nfor n in [1, 2, 3]:\n    total = n\nprint(total)",
        "explanation": "The loop overwrites total on each iteration instead of accumulating the values. Use total += n.",
        "fixed_code": "total = 0\nfor n in [1, 2, 3]:\n    total += n\nprint(total)"
    }
]

df = pd.DataFrame(examples)
df["prompt"] = df["bug"].apply(
    lambda code: f"Explain the bug in this Python code and suggest a fix:\n```python\n{code}\n```"
)
df["answer"] = df.apply(
    lambda row: (
        f"Explanation: {row['explanation']}\n\n"
        f"Corrected code:\n```python\n{row['fixed_code']}\n```"
    ),
    axis=1
)

print("Number of starter examples:", len(df))
display(df[["bug", "explanation", "fixed_code"]].head(5))

Number of starter examples: 10


,bug,explanation,fixed_code
0,"def add(a, b):\n return a - b",The function subtracts instead of adding. Repl...,"def add(a, b):\n return a + b"
1,"items = [1, 2, 3]\nprint(items[3])",Python lists use zero-based indexing. A three-...,"items = [1, 2, 3]\nprint(items[2])"
2,if x = 5:\n print('five'),A single equals sign assigns a value and canno...,if x == 5:\n print('five')
3,for i in range(5)\n print(i),The for statement is missing a colon at the en...,for i in range(5):\n print(i)
4,"def greet(name):\nprint('Hello', name)",The function body must be indented beneath the...,"def greet(name):\n print('Hello', name)"


# 5. Split and Save the Dataset

Split the examples into training and test subsets using a fixed random seed, then save both subsets as JSON Lines files. With only ten examples, the test set is very small.

In [ ]:
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42
)

train_df.to_json(
    "bug_train.jsonl",
    orient="records",
    lines=True,
    force_ascii=False
)

test_df.to_json(
    "bug_test.jsonl",
    orient="records",
    lines=True,
    force_ascii=False
)

print("Training examples:", len(train_df))
print("Test examples:", len(test_df))
print("Saved bug_train.jsonl and bug_test.jsonl")

Training examples: 8
Test examples: 2
Saved bug_train.jsonl and bug_test.jsonl


# 6. Load the Base Code Model

Load the Qwen2.5-Coder 0.5B Instruct tokenizer and causal language model. The model is the base model that will receive the LoRA adapter.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-Coder-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

model.config.use_cache = False

print("Tokenizer loaded successfully!")
print("Model loaded successfully!")
print("Device:", next(model.parameters()).device)
print("Model parameters:", f"{sum(p.numel() for p in model.parameters()):,}")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Tokenizer loaded successfully!
Model loaded successfully!
Device: cuda:0
Model parameters: 494,032,768


# 7. Configure LoRA Fine-Tuning

Set the Low-Rank Adaptation (LoRA) parameters and attach trainable adapter layers to selected attention projections. This reduces the number of parameters trained compared with full fine-tuning.

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 1,081,344 || all params: 495,114,112 || trainable%: 0.2184


# 8. Upgrade torchao if Required by the Runtime

This troubleshooting cell upgrades `torchao`, which may be needed if the installed version conflicts with PEFT or the current PyTorch runtime. If the package upgrade requires it, restart the Colab runtime and rerun the setup/model/LoRA cells in order before continuing.

In [ ]:
%pip install -q --upgrade torchao

# 9. Check Dependency Versions After Troubleshooting

Print the installed `torchao` and PEFT versions to confirm which versions the runtime is using.

In [ ]:
import importlib.metadata as md

print("torchao:", md.version("torchao"))
print("PEFT:", md.version("peft"))

torchao: 0.18.0
PEFT: 0.21.2


# 10. Format Examples as Chat Training Data

Read the saved train/test files, convert each example into a chat-template text sequence, and wrap the formatted rows as Hugging Face `Dataset` objects for training and evaluation.

In [ ]:
from datasets import Dataset
import pandas as pd

# Ensure the training data exists from Cell 5
train_df = pd.read_json("bug_train.jsonl", lines=True)
test_df = pd.read_json("bug_test.jsonl", lines=True)

def format_example(row):
    messages = [
        {
            "role": "system",
            "content": (
                "You are a programming assistant. Explain Python bugs clearly, "
                "describe why they occur, and provide corrected code."
            )
        },
        {
            "role": "user",
            "content": row["prompt"]
        },
        {
            "role": "assistant",
            "content": row["answer"]
        }
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

train_df["text"] = train_df.apply(format_example, axis=1)
test_df["text"] = test_df.apply(format_example, axis=1)

train_dataset = Dataset.from_pandas(
    train_df[["text"]],
    preserve_index=False
)

test_dataset = Dataset.from_pandas(
    test_df[["text"]],
    preserve_index=False
)

print("Training examples:", len(train_dataset))
print("Test examples:", len(test_dataset))
print("\nSample formatted example:\n")
print(train_dataset[0]["text"][:1500])

Training examples: 8
Test examples: 2

Sample formatted example:

<|im_start|>system
You are a programming assistant. Explain Python bugs clearly, describe why they occur, and provide corrected code.<|im_end|>
<|im_start|>user
Explain the bug in this Python code and suggest a fix:
```python
name = 'Alex'
print(Name)
```<|im_end|>
<|im_start|>assistant
Explanation: Python variable names are case-sensitive. Name and name are different identifiers.

Corrected code:
```python
name = 'Alex'
print(name)
```<|im_end|>



# 11. Tokenize the Dataset

Tokenize the formatted text to a maximum sequence length of 512 tokens and configure a causal-language-model data collator. Padding and truncation keep batches a consistent size.

In [ ]:
from transformers import DataCollatorForLanguageModeling

MAX_LENGTH = 512

def tokenize_example(example):
    return tokenizer(
        example["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length"
    )

tokenized_train = train_dataset.map(
    tokenize_example,
    batched=False,
    remove_columns=train_dataset.column_names
)

tokenized_test = test_dataset.map(
    tokenize_example,
    batched=False,
    remove_columns=test_dataset.column_names
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

print("Tokenization complete!")
print("Training rows:", len(tokenized_train))
print("Test rows:", len(tokenized_test))
print("Tokens per example:", len(tokenized_train[0]["input_ids"]))

Map:   0%|          | 0/8 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

Tokenization complete!
Training rows: 8
Test rows: 2
Tokens per example: 512


# 12. Configure the Trainer

Set the training hyperparameters and initialize the Hugging Face `Trainer`, including evaluation, checkpoint saving, and mixed-precision training settings.

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./bug_explanation_lora",
    num_train_epochs=5,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=1,
    save_strategy="epoch",
    eval_strategy="epoch",
    fp16=True,
    report_to="none",
    gradient_checkpointing=False,
    optim="adamw_torch",
    seed=42
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    processing_class=tokenizer,
    data_collator=data_collator
)

print("Trainer configured successfully!")

Trainer configured successfully!


# 13. Train the LoRA Adapter

Run the training loop and print the returned training metrics. Record these metrics in your report, but do not treat them as proof of general debugging accuracy.

In [ ]:
train_result = trainer.train()

print("Training finished!")
print("Training metrics:", train_result.metrics)

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 151643}.


Epoch,Training Loss,Validation Loss
1,2.901201,3.004468
2,2.853919,2.856081
3,2.864019,2.735216
4,2.576925,2.654466
5,2.460273,2.618334


Training finished!
Training metrics: {'train_runtime': 23.3554, 'train_samples_per_second': 1.713, 'train_steps_per_second': 0.428, 'total_flos': 44111395553280.0, 'train_loss': 2.7766929864883423, 'epoch': 5.0}


# 14. Save the Fine-Tuned Adapter

Save the trained LoRA adapter and tokenizer to a dedicated output folder. The adapter is separate from the base model, so the base model is still required to use it.

In [ ]:
import os

OUTPUT_DIR = "./bug_explanation_lora/final_adapter"
os.makedirs(OUTPUT_DIR, exist_ok=True)

model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("LoRA adapter saved to:", OUTPUT_DIR)
print("Saved files:")
print(os.listdir(OUTPUT_DIR))

LoRA adapter saved to: ./bug_explanation_lora/final_adapter
Saved files:
['tokenizer.json', 'tokenizer_config.json', 'README.md', 'chat_template.jinja', 'adapter_model.safetensors', 'adapter_config.json']


# 15. Try a Test Bug Example

Generate a response for a sample Python function containing a likely bug. Review the answer manually: this is a qualitative demonstration, not a comprehensive evaluation.

In [ ]:
import torch

model.eval()

test_bug = """def multiply(a, b):
    return a + b

print(multiply(3, 4))"""

messages = [
    {
        "role": "system",
        "content": (
            "You are a Python debugging assistant. Explain the bug clearly "
            "and provide corrected code."
        )
    },
    {
        "role": "user",
        "content": f"Explain the bug and fix this Python code:\n```python\n{test_bug}\n```"
    }
]

# Return a dictionary-like BatchEncoding
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True
)

# Move input tensors to the model's device
inputs = {key: value.to(model.device) for key, value in inputs.items()}

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=180,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

# Remove the prompt tokens and decode only the generated answer
prompt_length = inputs["input_ids"].shape[1]
new_tokens = outputs[0][prompt_length:]
answer = tokenizer.decode(new_tokens, skip_special_tokens=True)

print("BUGGY CODE:\n", test_bug)
print("\nASSISTANT RESPONSE:\n", answer)

BUGGY CODE:
 def multiply(a, b):
    return a + b

print(multiply(3, 4))

ASSISTANT RESPONSE:
 The bug in the code is that the function `multiply` is not returning the correct result. The correct result should be `7`, not `12`. The function is supposed to add two numbers, but it's actually subtracting them.

Here's the corrected code:

```python
def multiply(a, b):
    return a - b

print(multiply(3, 4))
```

This will correctly compute the sum of 3 and 4, which is 7.


# 16. Package the Adapter for Download

Create a ZIP archive of the saved adapter directory and download it from Colab. Keep the adapter archive private if it contains anything you do not want to publish.

In [ ]:
import shutil
from google.colab import files

zip_path = shutil.make_archive(
    "/content/bug_explanation_lora_adapter",
    "zip",
    OUTPUT_DIR
)

files.download(zip_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# 17. Launch the Gradio Web Interface

Create a small web UI where a user can paste Python code and receive a generated explanation and suggested fix. The public share link created by Gradio may be temporary. Treat model output as experimental and verify suggested fixes before using them.

In [ ]:
import torch
import gradio as gr

model.eval()

def explain_bug(code):
    if not code or not code.strip():
        return "Please enter some Python code first."

    messages = [
        {
            "role": "system",
            "content": (
                "You are a Python debugging assistant. Identify the likely bug, "
                "explain why it occurs, and provide corrected code. "
                "If the code is valid, say so rather than inventing a bug."
            )
        },
        {
            "role": "user",
            "content": f"Explain and fix this Python code:\n```python\n{code}\n```"
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    )

    inputs = {
        key: value.to(model.device)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    prompt_length = inputs["input_ids"].shape[1]
    response = tokenizer.decode(
        outputs[0][prompt_length:],
        skip_special_tokens=True
    )

    return response.strip()


demo = gr.Interface(
    fn=explain_bug,
    inputs=gr.Code(
        label="Enter Python code",
        language="python",
        lines=10
    ),
    outputs=gr.Markdown(label="Bug explanation and suggested fix"),
    title="Code Bug Explanation Assistant using LoRA",
    description=(
        "An experimental Python debugging assistant fine-tuned with "
        "Low-Rank Adaptation (LoRA)."
    ),
    examples=[
        ["def multiply(a, b):\n    return a + b\n\nprint(multiply(3, 4))"],
        ["numbers = [1, 2, 3]\nprint(numbers[3])"],
        ["for i in range(5)\n    print(i)"]
    ]
)

demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://dd8c287c29c411d35a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://dd8c287c29c411d35a.gradio.live
